### Colab Mount


In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    src_dir = Path("/content/drive/MyDrive/Research/AI-In-Science-Lab/local_learning/src")
else:
    cwd = Path.cwd().resolve()
    src_dir = cwd if (cwd / "models").is_dir() else cwd / "local_learning" / "src"

os.chdir(src_dir)
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))


### Setup


In [ ]:
import torch
import wandb

from training import train

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {DEVICE}")


### Experiment Configs


In [ ]:
BASE_CONFIG = {
    "project": "local-learning-3",
    "dataset": "cifar10",
    "preprocessing": "none",
    "training_mode": "classification",
    "loss_type": "regular",
    "learning_rate": 0.006,
    "epochs": 10,
    "batch_size": 64,
    "num_workers": 2,
    "lambda_strength": 0.0,
    "kernel_size": 3,
    "correlation_mode": "max",
    "comparison_mode": "shift",
    "normalization_mode": "relu",
    "postcomp_mode": "squared",
    "correlation_loss": "sum",
    "local": False,
    "local_training": False,
    "local_reconstruction_strength": 1.0,
    "hidden_ch": 48,
    "hidden_channels": [48, 32],
    "num_layers": 2,
    "k_spatial": 0.2,
    "k_population": None,
    "k_lifetime": 0.2,
    "wta_eval_multiplier": 1.0,
    "log_every_steps": 100,
    "log_weight_grids_every_steps": 100,
    "log_flops_every_steps": 0,
    "max_eval_batches": 25,
}

BASIC_CNN_CONFIG = {
    **BASE_CONFIG,
    "architecture_type": "cnn1",
}

GREEDY_POPULATION_CONFIG = {
    **BASE_CONFIG,
    "architecture_type": "greedy_stacked_autoencoder",
    "preprocessing": "whiten",
    "k_population": 0.2,
    "k_lifetime": None,
    "local_training": False,
}

GREEDY_POPULATION_LOCAL_CONFIG = {
    **GREEDY_POPULATION_CONFIG,
    "local_training": True,
    "local_reconstruction_strength": 1.0,
}

EXPERIMENT_CONFIGS = {
    "basic_cnn": BASIC_CNN_CONFIG,
    "greedy_population": GREEDY_POPULATION_CONFIG,
    "greedy_population_local": GREEDY_POPULATION_LOCAL_CONFIG,
}


### Sweep


In [ ]:
LOCAL_VS_GLOBAL_SWEEP_CONFIG = {
    "name": "greedy-local-vs-global-lifetime-grid-32",
    "project": "local-learning-3",
    "method": "grid",
    "metric": {
        "name": "test/acc",
        "goal": "maximize",
    },
    "parameters": {
        "project": {"value": "local-learning-3"},
        "dataset": {"value": "cifar10"},
        "preprocessing": {"value": "whiten"},
        "training_mode": {"value": "classification"},
        "loss_type": {"value": "regular"},
        "architecture_type": {"value": "greedy_stacked_autoencoder"},
        "epochs": {"value": 10},
        "batch_size": {"value": 64},
        "num_workers": {"value": 2},
        "lambda_strength": {"value": 0.0},
        "kernel_size": {"value": 3},
        "correlation_mode": {"value": "max"},
        "comparison_mode": {"value": "shift"},
        "normalization_mode": {"value": "relu"},
        "postcomp_mode": {"value": "squared"},
        "correlation_loss": {"value": "sum"},
        "local": {"value": False},
        "local_reconstruction_strength": {"value": 1.0},
        "k_lifetime": {"value": 0.2},
        "wta_eval_multiplier": {"value": 1.0},
        "log_every_steps": {"value": 100},
        "log_weight_grids_every_steps": {"value": 100},
        "log_flops_every_steps": {"value": 0},
        "max_eval_batches": {"value": 25},
        "local_training": {"values": [False, True]},
        "learning_rate": {"values": [0.006, 3e-3]},
        "hidden_channels": {"values": [48, 32]},
        "num_layers": {"values": [2, 3]},
        "k_spatial": {"values": [0.1, 0.2]},
        "k_population": {"value": None},
    },
}

LOCAL_VS_GLOBAL_SWEEP_RUNS = 32


def sweep_train() -> None:
    train(BASE_CONFIG, device=DEVICE)


def run_local_vs_global_sweep(*, sweep_id: str | None = None, count: int = LOCAL_VS_GLOBAL_SWEEP_RUNS) -> str:
    if sweep_id is None:
        sweep_id = wandb.sweep(
            LOCAL_VS_GLOBAL_SWEEP_CONFIG,
            project=LOCAL_VS_GLOBAL_SWEEP_CONFIG["project"],
        )
    wandb.agent(sweep_id, function=sweep_train, count=count)
    return sweep_id

### Run


In [ ]:
sweep_id = run_local_vs_global_sweep()
sweep_id
